In [2]:
%load_ext autoreload
%reload_ext autoreload
%autoreload 2

In [3]:
import os
from pathlib import Path
import bioacoustics_model_zoo as bmz
import csv
import lancedb
import pyarrow as pa
import soundfile as sf
import librosa
import numpy as np
import tempfile
import pandas as pd
import torch
from torch_geometric.data import HeteroData
from collections import defaultdict
from torch_geometric.utils import to_networkx
import matplotlib.pyplot as plt
import networkx as nx


2025-07-25 09:34:20.188416: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1753461260.204895 2870513 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1753461260.210019 2870513 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1753461260.223392 2870513 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1753461260.223405 2870513 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1753461260.223406 2870513 computation_placer.cc:177] computation placer alr

In [4]:
# Perch can only generate embeddings for audio files greater than 5 seconds. Therefore, loop any short audio files to make it atleast 5 seconds
def pad_short_clip(audio_path):
    target_duration_sec = 5
    samplerate=sf.info(audio_path).samplerate
    target_len = samplerate * target_duration_sec
    y, sr = librosa.load(audio_path, sr=samplerate)
    #pad if less than 5 seconds
    if len(y) < target_len:
        reps = int(np.ceil(target_len / len(y)))
        y = np.tile(y, reps)[:target_len]
    return np.asarray(y, dtype=np.float32), sr

In [5]:
def generate_embedding(audio_path):
    info = sf.info(audio_path)
    duration = info.frames / info.samplerate #faster than using librosa to load length
    if (duration < 5):
        formatted_wav, sample_rate = pad_short_clip(audio_path)
        #creates a new wav file of 5 seconds long to generate embedding and then immediately deletes it
        with tempfile.NamedTemporaryFile(suffix=".wav", delete=True) as tmp:
            # Write the array to the temp .wav file
            sf.write(tmp.name, formatted_wav, sample_rate)
            # Use the file path for embedding
            embedding = model.embed(tmp.name)
    # if >=5 seconds then embed directly
    else:
        embedding = model.embed(audio_path)
    return embedding, duration

In [6]:
###### STEP 1: PARSE lIKED SOUNDS & COLLECT DATA IN GENERAL #######
#order: 
#   1. parse the csv of liked & generate a hashmap where key is the filename & value is the entire dicitonary that would be the entry to lancedb
#   2. go through ALL wav files in directory & see if its filename matches the filename of the key in the hashmp from step 1. if so, then, insert it as a new column in the value dictionary.
#        -> save path to all wav files in array
#   3. once you have gone through ALL wav files in the direcotry & if any do not have a filepath associated with them, then remove them from the hashmap entirely

# === Step 1: Parse the CSV & build the hashmap ===
csv_file_path = "/home/s.kamboj.400/unzipped-music/mount/Liked Sounds/Location A Sand Forrest/Metadat -  Sandforest.csv"
filename_to_metadata = {}

with open(csv_file_path, newline='', encoding='utf-8') as csvfile:
    reader = csv.DictReader(csvfile, delimiter=';')
    for row in reader:
        filename = row["FileName"]
        filename_to_metadata[filename] = dict(row)
    # After parsing your CSV

#get all columns so that you can create empty frames later on
fieldnames = list(next(iter(filename_to_metadata.values())).keys())
fieldnames.append("FilePath") 
#print(fieldnames)


# === Step 2: Walk through all .wav files and insert filepath ===
root_dir = "/home/s.kamboj.400/unzipped-music/mount/"
matched_filenames = set()
#all_audio_files = []

countInvalid=0
countLessThanFive=0

for dirpath, _, filenames in os.walk(root_dir):
    for name in filenames:
        if name.endswith(".wav"):
            full_path = os.path.join(dirpath, name)
            #makes sure file is not corrupted
            try:
                sf.info(full_path)
            except (RuntimeError, sf.LibsndfileError):
                continue

            # duration_seconds = librosa.get_duration(path=full_path)
            # if (duration_seconds<5):
            #     countLessThanFive+=1
            if name in filename_to_metadata:
                # Case 1: metadata already exists for liked sounds — just add path
                filename_to_metadata[name]["FilePath"] = full_path
                matched_filenames.add(name)
            elif "Liked Sounds" in os.path.normpath(dirpath).split(os.sep):
                # Case 2: in "Liked Sounds" folder but not in the metadata csv filename — add blank metadata (this is because there are some files in liked sounds that are not listed in the metadata csv)
                    # we do still need to give it the metadata frame 
                new_entry = {field: "" for field in fieldnames}
                new_entry["FileName"] = name
                new_entry["FilePath"] = full_path
                filename_to_metadata[name] = new_entry
                #ensures this 
                matched_filenames.add(name)
            #else:
                #not a liked song at all. then, store its path so that you can generate and insert embeddings into lancedb. 
                # use the other audio paths in the frame to generate embeddings for queries
                #UNCOMMENT ALL_AUDIO_FILES TO USE ALL AUDIO FILES IN DTORY
                #all_audio_files.append(str(full_path))
            
                # print(f"This audio file {full_path} is not valid (probably corrupted), so nothing is happening")

# === Step 3: Remove unmatched entries ===
# This will only keep entries that were matched with a .wav file, basically deleted any "liked files" whose audio does not actually exist
filename_to_metadata = {
    fname: metadata
    for fname, metadata in filename_to_metadata.items()
    if fname in matched_filenames
}
# print("len of all audio files is ", len(all_audio_files))
# print("len of all metadata is ", len(filename_to_metadata))
# print(f"There are {countInvalid} invalid files")
# print(f"There are {countLessThanFive} audio recordings less than 5 seconds") #3464 audio recordings less than 5 seconds
#no longer make it a key-value pair. now metadata_list is a list of dictionaries of liked sounds that are ready to be inserted into lancedb
metadata_list = list(filename_to_metadata.values())
metadata_list

[{'FileName': 'SERRA_20250503_153307.wav',
  'Format': '256 16',
  'Note': 'Spaced Out Cicada, Bat in Between',
  'Take': 'Wet to Dry Season',
  'Scene': 'Sand Forrest (Location A)',
  'Project': 'NGS Gorongosa',
  'Category': 'Bats / Cicadas',
  'Library': 'Wet To Dry (May)',
  'Tape': '08 AM - 4PM',
  'Channels': '1',
  'Originator': 'Wildlife Accoustics',
  'Reference': 'Ultrasonic',
  'Description': '',
  'Duration': '0:04',
  'FilePath': '/home/s.kamboj.400/unzipped-music/mount/Location A (Sand Forrest)/LA_Sand_Forrest_ULTRASONIC_03-05-25/SERRA_20250503_153307.wav'},
 {'FileName': 'SERRA_20250503_153021.wav',
  'Format': '256 16',
  'Note': 'Cicada Groove',
  'Take': 'Wet to Dry Season',
  'Scene': 'Sand Forrest (Location A)',
  'Project': 'NGS Gorongosa',
  'Category': 'Bats / Cicadas',
  'Library': 'Wet To Dry (May)',
  'Tape': '08 AM - 4PM',
  'Channels': '1',
  'Originator': 'Wildlife Accoustics',
  'Reference': 'Ultrasonic',
  'Description': '',
  'Duration': '0:07',
  'FileP

In [ ]:
df = pd.DataFrame(metadata_list)
df.to_csv('liked-sounds-to-clean.csv', index=False)

In [7]:
# FORMAT DATA FROM LIKED-SOUNDS-CLEAN INTO A GRAPH
#load cleaned liked sounds csv
df = pd.read_csv("liked-sounds-clean.csv")  # replace with actual path
columns = df.columns.tolist()
audio_nodes = df["FileName"].tolist()
file_paths= df["FilePath"].tolist()
# # Generate vector embeddings of all audios using perch embeddings and insert into lancedb
model=bmz.Perch()

/home/s.kamboj.400/pyha-analyzer-2.0/.venv/lib/python3.11/site-packages/tensorflow_hub/__init__.py:61: DeprecationWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html
  from pkg_resources import parse_version
I0000 00:00:1753461304.839684 2870513 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 46762 MB memory:  -> device: 0, name: NVIDIA RTX A6000, pci bus id: 0000:02:00.0, compute capability: 8.6


MultilabelAveragePrecision init with num_labels  10932  and average  macro


/home/s.kamboj.400/pyha-analyzer-2.0/.venv/lib/python3.11/site-packages/opensoundscape/ml/cnn.py:599: UserWarning: 
                    This architecture is not listed in opensoundscape.ml.cnn_architectures.ARCH_DICT.
                    It will not be available for loading after saving the model with .save() (unless using pickle=True). 
                    To make it re-loadable, define a function that generates the architecture from arguments: (n_classes, n_channels) 
                    then use opensoundscape.ml.cnn_architectures.register_architecture() to register the generating function.

                    The function can also set the returned object's .constructor_name to the registered string key in ARCH_DICT
                    to avoid this warning and ensure it is reloaded correctly by opensoundscape.ml.load_model().

                    See opensoundscape.ml.cnn_architectures module for examples of constructor functions
                    
  warnings.warn(
/home/s.kambo

In [8]:
#initialize mapping to nodename and id
node_id_maps = defaultdict(dict) 
#keep list of nodes
node_lists = defaultdict(list) 
edge_lists = defaultdict(list)

In [9]:
audio_embeddings=[]
# Add audio nodes first
for idx, name in enumerate(audio_nodes):
    node_id_maps["audio"][name] = idx #map unique id to audio node name
    node_lists["audio"].append(name) #add name of audio node to the list's audio section
    #call generate vector embedding on currfilepath index and save embedding in array
    embedding = generate_embedding(file_paths[idx])  # shape: (embedding_dim,)
    audio_embeddings.append(embedding)

  0%|          | 0/1 [00:00<?, ?it/s]

I0000 00:00:1753461314.673133 2870513 service.cc:152] XLA service 0x59263569d070 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1753461314.673165 2870513 service.cc:160]   StreamExecutor device (0): NVIDIA RTX A6000, Compute Capability 8.6
2025-07-25 09:35:15.196395: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
2025-07-25 09:35:15.257869: W tensorflow/compiler/tf2xla/kernels/assert_op.cc:39] Ignoring Assert operator jax2tf_infer_fn_/assert_equal_1/Assert/AssertGuard/Assert
I0000 00:00:1753461316.088001 2870513 cuda_dnn.cc:529] Loaded cuDNN version 90501
2025-07-25 09:35:17.666697: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_51', 172 bytes spill stores, 172 bytes spill loads

2025-07-25 09:35:18.375371: I exter

  0%|          | 0/1 [00:00<?, ?it/s]

  0%|          | 0/1 [00:00<?, ?it/s]

  0%|          | 0/1 [00:00<?, ?it/s]

...

  0%|          | 0/51 [00:00<?, ?it/s]

  0%|          | 0/103 [00:00<?, ?it/s]

  0%|          | 0/117 [00:00<?, ?it/s]

  0%|          | 0/108 [00:00<?, ?it/s]

  0%|          | 0/126 [00:00<?, ?it/s]

In [10]:
# --- Step 3: Loop over each column and make edges ---
for col in columns:
    if col == "FileName" or col=="FilePath" or col=="Duration":
        continue
    for idx, row in df.iterrows():
        audio_name = row["FileName"]
        audio_id = node_id_maps["audio"][audio_name]
        
        # Handle multi-values separated by slashes
        raw_vals = str(row[col]).split("/")
        for raw_val in raw_vals:
            val = raw_val.strip()
            if val == "":
                continue

            # Register this node in the bigger list
            if val not in node_id_maps[col]:
                node_id_maps[col][val] = len(node_lists[col])
                node_lists[col].append(val)
            val_id = node_id_maps[col][val]

            # Add edge
            edge_lists[("audio", col, col)].append((audio_id, val_id))

In [11]:
# --- Step 4: Create HeteroData object ---
data = HeteroData()

# Add nodes
for node_type, names in node_lists.items():
    data[node_type].num_nodes = len(names)

# Add edges
for (src_type, rel_type, dst_type), edges in edge_lists.items():
    edge_index = torch.tensor(edges, dtype=torch.long).t().contiguous()
    data[(src_type, rel_type, dst_type)].edge_index = edge_index
    
audio_feat_matrix = np.vstack(audio_embeddings)
data["audio"].x = torch.tensor(audio_feat_matrix, dtype=torch.float)

data

ValueError: setting an array element with a sequence. The requested array has an inhomogeneous shape after 1 dimensions. The detected shape was (2,) + inhomogeneous part.

In [1]:
def hetero_to_networkx_subgraph_labeled(data, node_lists, audio_idx_param=0):
    G = nx.MultiDiGraph()
    
    # Helper to get human-readable label
    def get_label(node_type, idx):
        try:
            return node_lists[node_type][idx]
        except IndexError:
            return f"{node_type}_{idx}"
    for i in range(audio_idx_param):
        audio_idx = i
        audio_node = ('audio', audio_idx)
        G.add_node(audio_node, node_type='audio', label=get_label('audio', audio_idx))

        for edge_type in data.edge_types:
            src_type, rel_type, dst_type = edge_type
            edge_index = data[edge_type].edge_index

            for i in range(edge_index.size(1)):
                src = (src_type, int(edge_index[0, i]))
                dst = (dst_type, int(edge_index[1, i]))

                if src == audio_node:
                    G.add_node(dst, node_type=dst_type, label=get_label(dst_type, dst[1]))
                    G.add_edge(src, dst, rel_type=rel_type)
                elif dst == audio_node:
                    G.add_node(src, node_type=src_type, label=get_label(src_type, src[1]))
                    G.add_edge(src, dst, rel_type=rel_type)

    return G

# Create labeled subgraph
#audio_idx_param is number of audio files to include in graph visualization
G_sub = hetero_to_networkx_subgraph_labeled(data, node_lists, audio_idx_param=4)

# Draw with actual labels
plt.figure(figsize=(12, 10))
pos = nx.spring_layout(G_sub, seed=42)

color_map = []
for node, attrs in G_sub.nodes(data=True):
    ntype = attrs.get("node_type", "unknown")
    color_map.append({
        "audio": "red",
        "Category": "green",
        "Scene": "blue",
        "Season": "orange",
        "Time": "purple",
        "Note": "pink",
        "Reference": "gray",
        "Month": "brown",
        #"": "teal",
        "Tape": "yellow",
    }.get(ntype, "black"))

# Use the human-readable label
labels = {node: data['label'] for node, data in G_sub.nodes(data=True)}

nx.draw(
    G_sub, pos,
    node_color=color_map,
    node_size=500,
    with_labels=True,
    labels=labels,
    font_size=8,
    edge_color='gray',
    arrows=True
)

# Optional: label edge types
edge_labels = {(u, v): d['rel_type'] for u, v, d in G_sub.edges(data=True)}
nx.draw_networkx_edge_labels(G_sub, pos, edge_labels=edge_labels, font_size=7)

plt.title("Subgraph: Two Audio File and All Its Relationships")
plt.axis("off")
plt.show()


NameError: name 'data' is not defined

In [ ]:
####### STEP 2: insert vector embeddings of all sounds into lancedb #######
#Insert data into lancedb
count=0
records_to_insert=[]
batch_size_to_insert=100
# countOfLessThan5=0
for curr_wav_file in all_audio_files:
    #return embedding & duration so you can determine if looped or not
    embedding, duration= generate_embedding(curr_wav_file)

    #loop through all chunks of 5 second recordings for current wav file and insert into lancedb
    for i in range(embedding.shape[0]):
        embedding = np.array(embedding) # forces len of embedding to be 1280 by not letting embedding change dimensions
        start_sec = i * 5
        end_sec = (i + 1) * 5
        duration_str = f"{start_sec}-{end_sec}"
        metadata = {
            "FileName": os.path.basename(curr_wav_file),
            "Format": "",         
            "Note": "",
            "Take": "",
            "Scene": "",
            "Project": "",
            "Category": "",
            "Library": "",
            "Tape": "",
            "Channels": "",
            "Originator": "",
            "Reference": "",
            "Description": "",
            "Duration": duration_str,
            "FilePath": curr_wav_file,
            "Looped": duration < 5,
            "vector_embedding": embedding[i].tolist(),  
        }
        records_to_insert.append(metadata)
        #fast batching for lancedb insertion & memory safe
        if len(records_to_insert) >= batch_size_to_insert:
            table.add(records_to_insert)
            records_to_insert.clear()
    # count+=1
    # if (count>=5000):
    #     break
#insert any remaining records
if records_to_insert:
    table.add(records_to_insert)

In [ ]:
#check stuff was inserted
df = table.to_pandas()
print(df.head())  # Show first 5 rows

In [ ]:
###### STEP 3 : Generate vector embeddings of liked sounds and generate similarity search of liked sounds #######
#loop through liked sounds & generate vector embeddings
from IPython.display import Audio, display
count=0
for currLikedDict in metadata_list:
    audio_path= currLikedDict["FilePath"]
    embedding, _= generate_embedding(audio_path)
    for i in range(embedding.shape[0]):
        embedding = np.array(embedding)
        query_vector = embedding[i].tolist()
        #search lancedb with those vector embeddings
        results = table.search(query_vector).limit(1).to_pandas()
        print("The most similar to ", audio_path, " chunk #", (i+1), " is", results["FilePath"].tolist()[0])
        print("Liked audio:")
        display(Audio(audio_path))
        matched_path = results["FilePath"].tolist()[0]
        matched_duration = results["Duration"].tolist()[0]
        # Parse "start-end" from Duration field
        start_sec, end_sec = map(int, matched_duration.split('-'))
        duration = end_sec - start_sec

        # Load just the 5-second chunk (efficient)
        y, sr = librosa.load(matched_path, sr=None, offset=start_sec, duration=duration)

        # Display the audio player
        print("Most similar 5-second chunk from time :", matched_duration)
        display(Audio(y, rate=sr))

    # count+=1
    # if (count>5):
    #     break
